# Lookback Lens

Fit logistic regression to the per-head attention ratio between the prompt and previous reply tokens. The experiment adapts the method to token-level code defect detection. [Paper](https://arxiv.org/abs/2407.07071)

Train an independent probe on each benchmark. Line and sample scores use the maximum annotated token score. F1 thresholds come from validation.

In [1]:
from pathlib import Path
import sys

if Path("../src/traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
import torch
from IPython.display import display

from datasets.benchmarks import DATASETS, PILOT_LIMITS
from src.evaluation import TraceStore, evaluate
from src.methods import LookbackLens

pilot = False
limits = PILOT_LIMITS if pilot else None
torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)

In [3]:
results = []
for benchmark in DATASETS:
    splits = store.load_splits(benchmark, limits)
    results.extend(
        {"benchmark": benchmark, **row}
        for row in evaluate(LookbackLens(), splits)
    )

metrics = pd.DataFrame(results).set_index(["benchmark", "level"])
display(metrics.round(4))

n  positives  positive_rate  pr_auc  \
benchmark      level                                            
mbpp           token   3373        660         0.1957  0.5925   
               line     509         54         0.1061  0.7021   
               sample    74         31         0.4189  0.7318   
humaneval      token   3714        665         0.1791  0.5354   
               line     484         55         0.1136  0.5595   
               sample    46         19         0.4130  0.5042   
humaneval_java token   1157        205         0.1772  0.3021   
               line     155         15         0.0968  0.2903   
               sample    36         11         0.3056  0.3542   
defects4j      token   3901       2220         0.5691  0.9413   
               line     448        148         0.3304  0.8959   
               sample    86         35         0.4070  0.6586   

                       average_precision  roc_auc  precision  recall      f1  \
benchmark      level                                                           
mbpp           token              0.5938   0.8801     0.5272  0.7197  0.6086   
               line               0.7133   0.9652     0.6667  0.5926  0.6275   
               sample             0.7502   0.8627     0.5439  1.0000  0.7045   
humaneval      token              0.5368   0.8556     0.2366  0.9970  0.3825   
               line               0.5717   0.9096     0.2259  0.9818  0.3673   
               sample             0.5342   0.5302     0.3824  0.6842  0.4906   
humaneval_java token              0.3053   0.7321     0.2976  0.3659  0.3282   
               line               0.3196   0.8748     0.3043  0.4667  0.3684   
               sample             0.4016   0.6218     0.3750  0.5455  0.4444   
defects4j      token              0.9414   0.9153     0.7689  0.9144  0.8354   
               line               0.8962   0.9383     0.7792  0.8108  0.7947   
               sample             0.6636   0.6824     0.4462  0.8286  0.5800   

                       threshold  
benchmark      level              
mbpp           token      0.2727  
               line       0.9166  
               sample     0.7024  
humaneval      token      0.0049  
               line       0.1285  
               sample     0.5454  
humaneval_java token      0.1988  
               line       0.7120  
               sample     0.7120  
defects4j      token      0.2424  
               line       0.8277  
               sample     0.5828